### Trabajo Integrador Final - Python para la Ciencia de Datos

**Prefesores:**

- Acosta, Gabriel
- Benitez, Flavian Dante

---

**Estudiantes:**

- Silvera, Matías
- Ayala, Lautaro
- Gaona, Axel

---

Análisis y predicción de precio de reventa de vehículos usados.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [12]:
#Importamos el dataset y revisamos los primeros registros para entender su estructura y contenido
df = pd.read_csv('Car details v3.csv')
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [13]:
print("\nTipos de datos:")
print(df.dtypes)


Tipos de datos:
name                 str
year               int64
selling_price      int64
km_driven          int64
fuel                 str
seller_type          str
transmission         str
owner                str
mileage              str
engine               str
max_power            str
torque               str
seats            float64
dtype: object


In [ ]:
print("Forma del dataset:", df.shape)
# Verificamos cuantos 0 exactos tenemos en cada columna
(df == 0).sum()

De momento no hemos encontrado anomalías en las columnas

In [11]:
#Análisi de valores nulos en el dataset

datos_nulos = df.isnull().sum()
perdidos_pct = (datos_nulos / len(df) * 100).round(2)
perdidos_df = pd.DataFrame({
    'Cantidad de datos nulos': datos_nulos,
    '% Perdida': perdidos_pct
})
print(perdidos_df[perdidos_df['Cantidad de datos nulos'] > 0])

           Cantidad de datos nulos  % Perdida
mileage                        221       2.72
engine                         221       2.72
max_power                      215       2.65
torque                         222       2.73
seats                          221       2.72


**Hallazgo**: Se detectaron datos nulos en las columnas
- mileage
- engine
- max_power
- torque
- seats

**Decisión**: Se eliminarán los datos debido a que representan un porcentaje ínfimo del dataset y utilizar técnicas de imputación podría llevar el riesgo de introducir un sesgo en la varianza del modelo.

*Nota: Primero se creará una nueva columna con las marcas de los autos y se convertiran columnas clave a numéricas antes de proceder con la eliminación de nulos.*

In [15]:
# Verificación de estado de las columnas

for col in ["mileage", "engine", "max_power", "torque"]:
    if col in df.columns:
        print(f" {col}: {df[col].dropna().head(3).tolist()} ")

import re
def extraer_numero(value):
    if pd.isna(value):
        return np.nan
    val = str(value).strip().lower()

    match = re.search(r'([0-9]*\.?[0-9]+)', val)
    if match:
        return float(match.group(1))
    else:
        return np.nan

 mileage: ['23.4 kmpl', '21.14 kmpl', '17.7 kmpl'] 
 engine: ['1248 CC', '1498 CC', '1497 CC'] 
 max_power: ['74 bhp', '103.52 bhp', '78 bhp'] 
 torque: ['190Nm@ 2000rpm', '250Nm@ 1500-2500rpm', '12.7@ 2,700(kgm@ rpm)'] 


In [16]:
# Extraemos los números de las columnas problemáticas
for col in ["mileage", "engine", "max_power"]:
    if col in df.columns:
        df[col] = df[col].apply(extraer_numero)

# Verificamos los resultados
for col in ["mileage", "engine", "max_power"]:
    if col in df.columns:
        print(f" {col}: {df[col].dropna().head(3).tolist()} ")

 mileage: [23.4, 21.14, 17.7] 
 engine: [1248.0, 1498.0, 1497.0] 
 max_power: [74.0, 103.52, 78.0] 
